# Weather Data for the corn belt

[https://daymet.ornl.gov/](https://daymet.ornl.gov/)

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
from dotenv import load_dotenv
import time
from io import StringIO

## Load Yield data

In [3]:
# I need fips to be a string in case there are leading zeros
yield_df = pd.read_csv('./County Corn Yield Data/corn_belt_yield.csv', dtype={"fips": str})
yield_df.head()

,state_name,state_alpha,state_ansi,county_ansi,fips,county_name,year,short_desc,unit_desc,statisticcat_desc,Value
0,ILLINOIS,IL,17,11,17011,BUREAU,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.9
1,ILLINOIS,IL,17,11,17011,BUREAU,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,240.8
2,ILLINOIS,IL,17,11,17011,BUREAU,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,223.5
3,ILLINOIS,IL,17,11,17011,BUREAU,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.1
4,ILLINOIS,IL,17,11,17011,BUREAU,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,203.5


## Checking Data

In [4]:
yield_df.shape

(24644, 11)

In [5]:
yield_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24644 entries, 0 to 24643
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   state_name         24644 non-null  object 
 1   state_alpha        24644 non-null  object 
 2   state_ansi         24644 non-null  int64  
 3   county_ansi        24644 non-null  int64  
 4   fips               24644 non-null  object 
 5   county_name        24644 non-null  object 
 6   year               24644 non-null  int64  
 7   short_desc         24644 non-null  object 
 8   unit_desc          24644 non-null  object 
 9   statisticcat_desc  24644 non-null  object 
 10  Value              24644 non-null  float64
dtypes: float64(1), int64(3), object(7)
memory usage: 2.1+ MB


In [6]:
yield_df.describe()

,state_ansi,county_ansi,year,Value
count,24644.000000,24644.000000,24644.000000,24644.000000
mean,28.079127,93.939012,2011.815533,147.829005
std,10.878044,57.255843,7.443268,38.241255
min,17.000000,1.000000,2000.000000,0.000000
25%,19.000000,45.000000,2005.000000,124.000000
50%,26.000000,91.000000,2011.000000,151.800000
75%,31.000000,139.000000,2018.000000,175.300000
max,55.000000,239.000000,2025.000000,253.600000


In [7]:
yield_df.describe(include='object')

,state_name,state_alpha,fips,county_name,short_desc,unit_desc,statisticcat_desc
count,24644,24644,24644,24644,24644,24644,24644
unique,13,13,1124,704,1,1,1
top,IOWA,IA,17011,JACKSON,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD
freq,2470,2470,26,231,24644,24644,24644


In [8]:
yield_df.isnull().sum()

state_name           0
state_alpha          0
state_ansi           0
county_ansi          0
fips                 0
county_name          0
year                 0
short_desc           0
unit_desc            0
statisticcat_desc    0
Value                0
dtype: int64

In [9]:
yield_df.duplicated().sum()

0

In [10]:
for i in yield_df.select_dtypes(include='object').columns:
    print(yield_df[i].value_counts())
    print("*"*30)

state_name
IOWA            2470
ILLINOIS        2417
INDIANA         2142
KENTUCKY        2092
KANSAS          2090
NEBRASKA        2069
OHIO            2051
MISSOURI        1967
MINNESOTA       1867
WISCONSIN       1577
MICHIGAN        1485
SOUTH DAKOTA    1363
NORTH DAKOTA    1054
Name: count, dtype: int64
******************************
state_alpha
IA    2470
IL    2417
IN    2142
KY    2092
KS    2090
NE    2069
OH    2051
MO    1967
MN    1867
WI    1577
MI    1485
SD    1363
ND    1054
Name: count, dtype: int64
******************************
fips
17011    26
27169    26
26161    26
26147    26
26115    26
         ..
29055     1
29119     1
29229     1
29209     1
21189     1
Name: count, Length: 1124, dtype: int64
******************************
county_name
JACKSON       231
WASHINGTON    217
JEFFERSON     199
CLAY          196
MONROE        184
             ... 
MAGOFFIN        1
STONE           1
DENT            1
OWSLEY          1
ELLIOTT         1
Name: count, Length: 704, dty

## Latitude and Longitude of each County

To get the weather from Daymet, I need a latitude and longitude which I can get for each county to then apply the weather data to. 

[https://simplemaps.com/data/us-counties](https://simplemaps.com/data/us-counties)

In [11]:
coords_df = pd.read_csv('./County Corn Yield Data/uscounties.csv', dtype={"county_fips": str})
coords_df.head()

,county,county_ascii,county_full,county_fips,state_id,state_name,lat,lng,population
0,Los Angeles,Los Angeles,Los Angeles County,06037,CA,California,34.3219,-118.2247,9808667
1,Cook,Cook,Cook County,17031,IL,Illinois,41.8401,-87.8168,5182090
2,Harris,Harris,Harris County,48201,TX,Texas,29.8578,-95.3938,4838303
3,Maricopa,Maricopa,Maricopa County,04013,AZ,Arizona,33.3490,-112.4915,4559748
4,San Diego,San Diego,San Diego County,06073,CA,California,33.0343,-116.7350,3288774


In [12]:
coords_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3144 entries, 0 to 3143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   county        3144 non-null   object 
 1   county_ascii  3144 non-null   object 
 2   county_full   3144 non-null   object 
 3   county_fips   3144 non-null   object 
 4   state_id      3144 non-null   object 
 5   state_name    3144 non-null   object 
 6   lat           3144 non-null   float64
 7   lng           3144 non-null   float64
 8   population    3144 non-null   int64  
dtypes: float64(2), int64(1), object(6)
memory usage: 221.2+ KB


In [13]:
coords_df = coords_df[["county_fips", "lat", "lng"]]
coords_df.head()

,county_fips,lat,lng
0,06037,34.3219,-118.2247
1,17031,41.8401,-87.8168
2,48201,29.8578,-95.3938
3,04013,33.3490,-112.4915
4,06073,33.0343,-116.7350


In [14]:
coords_df = coords_df.rename(columns={"county_fips": "fips", "lat":"latitude", "lng":"longitude"})
coords_df.head()

,fips,latitude,longitude
0,06037,34.3219,-118.2247
1,17031,41.8401,-87.8168
2,48201,29.8578,-95.3938
3,04013,33.3490,-112.4915
4,06073,33.0343,-116.7350


### Merge the yield dataset with the coordinates

In [15]:
yield_df = yield_df.merge(coords_df, on="fips", how="left")
yield_df.head()

,state_name,state_alpha,state_ansi,county_ansi,fips,county_name,year,short_desc,unit_desc,statisticcat_desc,Value,latitude,longitude
0,ILLINOIS,IL,17,11,17011,BUREAU,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.9,41.4041,-89.5286
1,ILLINOIS,IL,17,11,17011,BUREAU,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,240.8,41.4041,-89.5286
2,ILLINOIS,IL,17,11,17011,BUREAU,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,223.5,41.4041,-89.5286
3,ILLINOIS,IL,17,11,17011,BUREAU,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,225.1,41.4041,-89.5286
4,ILLINOIS,IL,17,11,17011,BUREAU,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,203.5,41.4041,-89.5286


In [16]:
yield_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24644 entries, 0 to 24643
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   state_name         24644 non-null  object 
 1   state_alpha        24644 non-null  object 
 2   state_ansi         24644 non-null  int64  
 3   county_ansi        24644 non-null  int64  
 4   fips               24644 non-null  object 
 5   county_name        24644 non-null  object 
 6   year               24644 non-null  int64  
 7   short_desc         24644 non-null  object 
 8   unit_desc          24644 non-null  object 
 9   statisticcat_desc  24644 non-null  object 
 10  Value              24644 non-null  float64
 11  latitude           24644 non-null  float64
 12  longitude          24644 non-null  float64
dtypes: float64(3), int64(3), object(7)
memory usage: 2.4+ MB


### Make county locations without all of the years because Daymet returns all of the years together

In [17]:
county_locations = yield_df[["fips", "latitude", "longitude"]].drop_duplicates(subset="fips").reset_index(drop=True)
county_locations.head()

,fips,latitude,longitude
0,17011,41.4041,-89.5286
1,17015,42.0687,-89.9343
2,17073,41.3531,-90.1314
3,17085,42.3658,-90.2126
4,17103,41.7461,-89.3003


## Get weather data from Daymet

I am using the [Single Pixel Extraction Tool](https://daymet.ornl.gov/single-pixel-tool-guide).

In [18]:
DAYMET_URL = "https://daymet.ornl.gov/single-pixel/api/data"

In [19]:
# Make directory to save each of the weather files in
os.makedirs("daymet_counties", exist_ok=True)

In [20]:
# all_weather_data = []

for index, row in county_locations.iterrows():

    county_file = f"daymet_counties/daymet_{row["fips"]}.csv"

    # Skip the county if its file already exists.
    if os.path.exists(county_file):
        print(f"{index}/{len(county_locations)}: {row["fips"]} already completed")
        continue

    print(f"Downloading county {index + 1} of {len(county_locations)}:", row["fips"])
    
    parameters = {
        "lat": row["latitude"],
        "lon": row["longitude"],
        "vars": "tmax,tmin,srad,vp,swe,prcp,dayl",
        "start": "2000-01-01",
        "end": "2025-12-31"
    }

    try:

        r = requests.get(
            DAYMET_URL,
            params=parameters,
            timeout=120
        )
    
        r.raise_for_status()
        
        data = pd.read_csv(StringIO(r.text), skiprows=6)
    
        data["fips"] = row["fips"]
        data["latitude"] = row["latitude"]
        data["longitude"] = row["longitude"]
    
        # print(type(data))
        # print(data)
        # print("*"*30)

        data.to_csv(county_file, index=False)
        
        print("Saved")

    except Exception as error:
        print("Failed:", error)
        
    # all_weather_data.append(data)

    time.sleep(0.1)

0/1124: 17011 already completed
1/1124: 17015 already completed
2/1124: 17073 already completed
3/1124: 17085 already completed
4/1124: 17103 already completed
5/1124: 17131 already completed
6/1124: 17141 already completed
7/1124: 17155 already completed
8/1124: 17161 already completed
9/1124: 17177 already completed
10/1124: 17195 already completed
11/1124: 17201 already completed
12/1124: 17007 already completed
13/1124: 17031 already completed
14/1124: 17037 already completed
15/1124: 17043 already completed
16/1124: 17063 already completed
17/1124: 17089 already completed
18/1124: 17093 already completed
19/1124: 17097 already completed
20/1124: 17099 already completed
21/1124: 17111 already completed
22/1124: 17197 already completed
23/1124: 17001 already completed
24/1124: 17009 already completed
25/1124: 17057 already completed
26/1124: 17067 already completed
27/1124: 17071 already completed
28/1124: 17095 already completed
29/1124: 17109 already completed
30/1124: 17169 alrea

### Combine all lof the weather files

In [21]:
all_weather_data = []

for filename in os.listdir("daymet_counties"):
    if filename.endswith(".csv"):
        data = pd.read_csv(f"daymet_counties/{filename}", dtype={"fips": str})
        all_weather_data.append(data)

In [22]:
weather_df = pd.concat(all_weather_data, ignore_index=True)
weather_df.head()

,year,yday,dayl (s),prcp (mm/day),srad (W/m^2),swe (kg/m^2),tmax (deg c),tmin (deg c),vp (Pa),fips,latitude,longitude
0,2000,1,30470.40,13.66,125.50,111.96,-1.81,-11.57,252.27,26103,46.4313,-87.6416
1,2000,2,30524.32,15.97,67.20,127.93,-1.97,-6.53,374.84,26103,46.4313,-87.6416
2,2000,3,30582.63,3.26,52.46,131.19,-3.06,-6.49,375.82,26103,46.4313,-87.6416
3,2000,4,30645.28,2.16,76.78,133.35,-5.59,-10.71,270.14,26103,46.4313,-87.6416
4,2000,5,30712.22,0.00,227.57,133.35,-5.69,-23.06,95.43,26103,46.4313,-87.6416


In [23]:
weather_df.tail()

,year,yday,dayl (s),prcp (mm/day),srad (W/m^2),swe (kg/m^2),tmax (deg c),tmin (deg c),vp (Pa),fips,latitude,longitude
10666755,2025,361,31357.98,1.45,46.35,25.35,2.11,0.21,619.97,55137,44.1131,-89.2429
10666756,2025,362,31386.34,13.63,50.90,23.34,1.83,-1.21,559.15,55137,44.1131,-89.2429
10666757,2025,363,31418.83,0.00,91.19,23.34,-5.28,-9.34,301.20,55137,44.1131,-89.2429
10666758,2025,364,31455.43,1.78,177.55,25.12,-5.17,-14.04,206.45,55137,44.1131,-89.2429
10666759,2025,365,31496.12,1.00,213.06,26.12,-1.91,-14.17,204.16,55137,44.1131,-89.2429


In [24]:
weather_df.columns

Index(['year', 'yday', 'dayl (s)', 'prcp (mm/day)', 'srad (W/m^2)',
       'swe (kg/m^2)', 'tmax (deg c)', 'tmin (deg c)', 'vp (Pa)', 'fips',
       'latitude', 'longitude'],
      dtype='object')

### What do these mean?

**`dayl`** ( Day Length (seconds/day (s/day) ) ) - Duration of the daylight period for the day. This calculation is based on the period of the day during which the sun is above a hypothetical flat horizon.

**`prcp`** ( Precipitation (millimeters/day (mm/day) ) ) - Daily total precipitation, sum of all forms converted to water-equivalent.

**`srad`** ( Shortwave Solar Radiation (watts per square meter (W/m²) ) ) - Amount of incoming shortwave solar energy reaching the surface. Daymet reports the **average radiation flux during daylight hours**, rather than an average over the entire 24-hour day.

**`swe`** ( Snow Water Equivalent (kilograms per square meter (kg/m²) ) ) - Snow water equivalent. The amount of water contained within the snowpack.

**`tmax`** ( Maximum Air Temperature (degrees Celsius (°C) ) ) - Daily maximum 2-meter air temperature.

**`tmin`** ( Minimum Air Temperature (degrees Celsius (°C) ) ) - Daily minimum 2-meter air temperature.

**`vp`** ( Water Vapor Pressure (Pascals (Pa) ) ) - Water Vapor Pressure. Daily average partial pressure of water vapor.

**Note:** Daily Total Radiation (MJ/m^2/day) can be calculated: ((srad (W/m^2) * dayl (s/day)) / l,000,000)

In [25]:
weather_df.shape

(10666760, 12)

In [26]:
# rename the columns
weather_df = weather_df.rename(columns={
    'dayl (s)': 'dayl_s', 
    'prcp (mm/day)': 'prcp_mm_day', 
    'srad (W/m^2)': 'srad_w_m2',
    'swe (kg/m^2)': 'swe_kg_m2', 
    'tmax (deg c)': 'tmax_c', 
    'tmin (deg c)': 'tmin_c', 
    'vp (Pa)': 'vp_pa'
}) 

In [27]:
weather_df["date"] = (pd.to_datetime(weather_df["year"], format="%Y") + pd.to_timedelta(weather_df["yday"] - 1, unit="D"))
weather_df["month"] = weather_df["date"].dt.month

In [28]:
weather_df.head()

,year,yday,dayl_s,prcp_mm_day,srad_w_m2,swe_kg_m2,tmax_c,tmin_c,vp_pa,fips,latitude,longitude,date,month
0,2000,1,30470.40,13.66,125.50,111.96,-1.81,-11.57,252.27,26103,46.4313,-87.6416,2000-01-01,1
1,2000,2,30524.32,15.97,67.20,127.93,-1.97,-6.53,374.84,26103,46.4313,-87.6416,2000-01-02,1
2,2000,3,30582.63,3.26,52.46,131.19,-3.06,-6.49,375.82,26103,46.4313,-87.6416,2000-01-03,1
3,2000,4,30645.28,2.16,76.78,133.35,-5.59,-10.71,270.14,26103,46.4313,-87.6416,2000-01-04,1
4,2000,5,30712.22,0.00,227.57,133.35,-5.69,-23.06,95.43,26103,46.4313,-87.6416,2000-01-05,1


In [29]:
# Calculate Solar Radiation
weather_df["radiation_mj_m2_day"] = weather_df["srad_w_m2"] * weather_df["dayl_s"] / 1_000_000

In [30]:
monthly_weather = weather_df.groupby(["fips", "year", "month"],as_index=False).agg(
    prcp_total_mm=("prcp_mm_day", "sum"),
    radiation_total_mj_m2=("radiation_mj_m2_day", "sum"),
    dayl_mean_s=("dayl_s", "mean"),
    swe_mean_kg_m2=("swe_kg_m2", "mean"),
    tmax_mean_c=("tmax_c", "mean"),
    tmin_mean_c=("tmin_c", "mean"),
    vp_mean_pa=("vp_pa", "mean"),
)

In [31]:
monthly_weather.head()

,fips,year,month,prcp_total_mm,radiation_total_mj_m2,dayl_mean_s,swe_mean_kg_m2,tmax_mean_c,tmin_mean_c,vp_mean_pa
0,17001,2000,1,19.61,233.077019,34257.320645,5.646774,2.590645,-7.356452,380.588065
1,17001,2000,2,79.19,298.927736,37762.033448,1.913103,9.046207,-1.495862,572.198276
2,17001,2000,3,46.96,447.767132,42394.024516,0.000000,13.606774,1.701935,703.285806
3,17001,2000,4,33.59,568.288936,47234.612333,0.000000,18.121667,4.885667,840.521000
4,17001,2000,5,77.09,639.980684,51282.445484,0.000000,24.685806,12.448065,1381.315484


In [32]:
# Turn each month into a column
monthly_weather = monthly_weather.pivot(
    index=["fips", "year"],
    columns="month"
)

In [33]:
monthly_weather.head()

prcp_total_mm                                                \
month                 1      2      3       4       5       6       7    
fips  year                                                               
17001 2000         19.61  79.19  46.96   33.59   77.09  196.25   85.81   
      2001         84.65  91.46  46.37   82.85  145.02  110.94   54.68   
      2002         78.13  29.34  35.35  138.66  238.94   97.42  123.64   
      2003         33.10  36.36  53.15   99.04   86.30  108.88  124.66   
      2004         18.83  18.87  80.74   62.27  129.39   76.41   78.78   

                                   ...  vp_mean_pa                            \
month           8      9       10  ...          3            4            5    
fips  year                         ...                                         
17001 2000  120.91  60.52   62.42  ...  703.285806   840.521000  1381.315484   
      2001  134.37  84.49  106.29  ...  496.050645  1115.226000  1493.618710   
      2002  118.20  13.96   88.94  ...  508.791935   992.366000  1203.445484   
      2003  132.89  98.73   48.05  ...  560.998710   924.592667  1280.123226   
      2004  229.61  28.77  163.48  ...  684.761613   890.752667  1496.920323   

                                                                             \
month                6            7            8            9            10   
fips  year                                                                    
17001 2000  1766.777000  2139.278065  2218.254839  1550.184000  1150.944516   
      2001  1819.425000  2299.301935  2048.718065  1451.843000   987.571935   
      2002  2065.985667  2358.078710  2131.864194  1593.893333   942.513548   
      2003  1692.230000  2137.109677  2155.233871  1342.730333   977.849677   
      2004  1788.652333  2016.278710  1671.298065  1501.030333  1053.663871   

                                    
month               11          12  
fips  year                          
17001 2000  580.671333  240.422667  
      2001  865.225667  547.358710  
      2002  581.450000  438.044194  
      2003  685.048667  482.509677  
      2004  764.416667  438.501667  

[5 rows x 84 columns]

In [34]:
monthly_weather.columns = [
    f"{weather_variable}_m{month:02d}"
    for weather_variable, month
    in monthly_weather.columns
]

In [35]:
monthly_weather.head()

prcp_total_mm_m01  prcp_total_mm_m02  prcp_total_mm_m03  \
fips  year                                                            
17001 2000              19.61              79.19              46.96   
      2001              84.65              91.46              46.37   
      2002              78.13              29.34              35.35   
      2003              33.10              36.36              53.15   
      2004              18.83              18.87              80.74   

            prcp_total_mm_m04  prcp_total_mm_m05  prcp_total_mm_m06  \
fips  year                                                            
17001 2000              33.59              77.09             196.25   
      2001              82.85             145.02             110.94   
      2002             138.66             238.94              97.42   
      2003              99.04              86.30             108.88   
      2004              62.27             129.39              76.41   

            prcp_total_mm_m07  prcp_total_mm_m08  prcp_total_mm_m09  \
fips  year                                                            
17001 2000              85.81             120.91              60.52   
      2001              54.68             134.37              84.49   
      2002             123.64             118.20              13.96   
      2003             124.66             132.89              98.73   
      2004              78.78             229.61              28.77   

            prcp_total_mm_m10  ...  vp_mean_pa_m03  vp_mean_pa_m04  \
fips  year                     ...                                   
17001 2000              62.42  ...      703.285806      840.521000   
      2001             106.29  ...      496.050645     1115.226000   
      2002              88.94  ...      508.791935      992.366000   
      2003              48.05  ...      560.998710      924.592667   
      2004             163.48  ...      684.761613      890.752667   

            vp_mean_pa_m05  vp_mean_pa_m06  vp_mean_pa_m07  vp_mean_pa_m08  \
fips  year                                                                   
17001 2000     1381.315484     1766.777000     2139.278065     2218.254839   
      2001     1493.618710     1819.425000     2299.301935     2048.718065   
      2002     1203.445484     2065.985667     2358.078710     2131.864194   
      2003     1280.123226     1692.230000     2137.109677     2155.233871   
      2004     1496.920323     1788.652333     2016.278710     1671.298065   

            vp_mean_pa_m09  vp_mean_pa_m10  vp_mean_pa_m11  vp_mean_pa_m12  
fips  year                                                                  
17001 2000     1550.184000     1150.944516      580.671333      240.422667  
      2001     1451.843000      987.571935      865.225667      547.358710  
      2002     1593.893333      942.513548      581.450000      438.044194  
      2003     1342.730333      977.849677      685.048667      482.509677  
      2004     1501.030333     1053.663871      764.416667      438.501667  

[5 rows x 84 columns]

In [36]:
monthly_weather = monthly_weather.reset_index()

In [37]:
monthly_weather.head()

,fips,year,prcp_total_mm_m01,prcp_total_mm_m02,prcp_total_mm_m03,prcp_total_mm_m04,prcp_total_mm_m05,prcp_total_mm_m06,prcp_total_mm_m07,prcp_total_mm_m08,...,vp_mean_pa_m03,vp_mean_pa_m04,vp_mean_pa_m05,vp_mean_pa_m06,vp_mean_pa_m07,vp_mean_pa_m08,vp_mean_pa_m09,vp_mean_pa_m10,vp_mean_pa_m11,vp_mean_pa_m12
0,17001,2000,19.61,79.19,46.96,33.59,77.09,196.25,85.81,120.91,...,703.285806,840.521000,1381.315484,1766.777000,2139.278065,2218.254839,1550.184000,1150.944516,580.671333,240.422667
1,17001,2001,84.65,91.46,46.37,82.85,145.02,110.94,54.68,134.37,...,496.050645,1115.226000,1493.618710,1819.425000,2299.301935,2048.718065,1451.843000,987.571935,865.225667,547.358710
2,17001,2002,78.13,29.34,35.35,138.66,238.94,97.42,123.64,118.20,...,508.791935,992.366000,1203.445484,2065.985667,2358.078710,2131.864194,1593.893333,942.513548,581.450000,438.044194
3,17001,2003,33.10,36.36,53.15,99.04,86.30,108.88,124.66,132.89,...,560.998710,924.592667,1280.123226,1692.230000,2137.109677,2155.233871,1342.730333,977.849677,685.048667,482.509677
4,17001,2004,18.83,18.87,80.74,62.27,129.39,76.41,78.78,229.61,...,684.761613,890.752667,1496.920323,1788.652333,2016.278710,1671.298065,1501.030333,1053.663871,764.416667,438.501667


### Save to .csv

In [38]:
weather_df.to_csv('corn_belt_weather.csv', index=False)

---

## Merge with corn yield data

In [39]:
yield_df["fips"] = (yield_df["fips"].astype(str).str.zfill(5))

monthly_weather["fips"] = (monthly_weather["fips"].astype(str).str.zfill(5))

yield_df["year"] = yield_df["year"].astype(int)
monthly_weather["year"] = monthly_weather["year"].astype(int)

In [40]:
final_df = yield_df.merge(
    monthly_weather,
    on=["fips", "year"],
    how="left",
    validate="one_to_one"
)

In [41]:
print("Yield rows:", len(yield_df))
print("Final rows:", len(final_df))

Yield rows: 24644
Final rows: 24644


## Save to .csv

In [42]:
final_df.to_csv('corn_belt_yield_weather.csv', index=False)